In [ ]:
### HBV_CALIBRATION

In [ ]:
import pandas as pd
import numpy as np
import spotpy
import os

# -------------------------
# 1️⃣ Define catchments
# -------------------------
catchments = [
    "E334602109",
    "E334602110",
    "E334602111",
    # Add more catchment IDs here
]

# Folder where CSVs are stored
data_folder = "data/"

# Output lists
results_list = []

# -------------------------
# 2️⃣ HBV + SPOTPY Setup (same as before)
# -------------------------
def run_hbv_rsm(P, T, PET, params):
    n = len(P)
    snow, liquid = 0.0, 0.0
    soil = params["FC"] * 0.5
    SUZ, SLZ = 0.0, 0.0
    Q_sim = np.zeros(n)
    
    for t in range(n):
        if T[t] <= params["TT"] - params["TTInt"]/2:
            frac_rain = 0.0
        elif T[t] >= params["TT"] + params["TTInt"]/2:
            frac_rain = 1.0
        else:
            frac_rain = (T[t] - (params["TT"] - params["TTInt"]/2)) / params["TTInt"]

        rainfall = P[t] * frac_rain
        snowfall = P[t] * (1 - frac_rain)

        snow += snowfall
        melt = min(params["CFMAX"] * max(T[t]-params["TT"], 0), snow)
        refreeze = min(params["CFR"]*params["CFMAX"]*max(params["TT"]-T[t],0), liquid)
        snow = snow - melt + refreeze
        liquid = liquid + rainfall + melt - refreeze
        water_to_soil = max(0, liquid - params["CWH"]*snow)
        liquid -= water_to_soil

        recharge = water_to_soil * (soil / params["FC"]) ** params["BETA"]
        soil += water_to_soil - recharge

        evap = PET[t] if soil > params["PWP"] else PET[t]*(soil/params["PWP"])
        soil = np.clip(soil-evap, 0, params["FC"])

        SUZ += recharge
        excess = max(0, SUZ - params["SUMax"])
        SUZ -= excess

        Qr = params["Kr"]*excess
        Qu = params["Ku"]*SUZ
        perc = min(params["Kperc"]*SUZ, SUZ)
        SUZ -= (Qu+perc)
        SLZ += perc
        Ql = params["Kl"]*SLZ
        SLZ -= Ql

        Q_sim[t] = Qr + Qu + Ql

    MAXBAS = int(params.get("MAXBAS",3))
    weights = np.arange(1, MAXBAS+1)
    weights = weights / weights.sum()
    Q_sim = np.convolve(Q_sim, weights, mode='full')[:len(Q_sim)]

    return Q_sim

# -------------------------
# 3️⃣ Loop Over Catchments
# -------------------------
all_results = []

for catch in catchments:
    print(f"Calibrating catchment {catch}...")

    df = pd.read_csv(os.path.join(data_folder, f"CAMELS_CH_obs_based_{catch}.csv"), parse_dates=["date"])
    
    # Split periods
    df_cal = df[(df["date"] >= "1982-01-01") & (df["date"] <= "2000-12-31")].reset_index(drop=True)
    df_val = df[(df["date"] >= "2010-01-01") & (df["date"] <= "2019-12-31")].reset_index(drop=True)

    warmup = 365

    # SPOTPY Setup
    class HBVSetupCatchment(object):
        def __init__(self, df):
            self.df = df
            self.P = df["precipitation(mm/d)"].values
            self.T = df["temperature_mean(degC)"].values
            self.PET = df["pet_sim(mm/d)"].values
            self.Q_obs = df["discharge_spec(mm/d)"].values
            self.warmup = warmup

        def parameters(self):
            return spotpy.parameter.generate([
                spotpy.parameter.Uniform('CFMAX', 1, 8),
                spotpy.parameter.Uniform('CFR', 0, 0.1),
                spotpy.parameter.Uniform('CWH', 0.01, 0.2),

                spotpy.parameter.Uniform('TT', -2, 3),
                spotpy.parameter.Uniform('TTInt', 0.5, 3),

                spotpy.parameter.Uniform('BETA', 1, 5),
                spotpy.parameter.Uniform('FC', 20, 500),
                spotpy.parameter.Uniform('PWP', 0.1, 1.0),

                spotpy.parameter.Uniform('SUMax', 10, 200),

                spotpy.parameter.Uniform('Kr', 0.01, 0.2),
                spotpy.parameter.Uniform('Ku', 0.01, 0.5),
                spotpy.parameter.Uniform('Kl', 0.001, 0.1),

                spotpy.parameter.Uniform('Kperc', 0.01, 1.0),

                spotpy.parameter.Uniform('MAXBAS', 1, 7)
            ])

        def simulation(self, vector):
            params = {k: vector[k] for k in vector.dtype.names}
            Q_sim = run_hbv_rsm(self.P, self.T, self.PET, params)
            return Q_sim[self.warmup:]

        def evaluation(self):
            return self.Q_obs[self.warmup:]

        def objectivefunction(self, simulation, evaluation):
            sim = np.array(simulation)
            obs = np.array(evaluation)
            return 1 - np.sum((obs-sim)**2)/np.sum((obs-np.mean(obs))**2)

    # Initialize SPOTPY
    setup = HBVSetupCatchment(df_cal)
    sampler = spotpy.algorithms.sceua(setup, dbname=f'hbv_cal_{catch}', dbformat='csv')

    # Run calibration
    sampler.sample(2000)  # reduce for testing, increase for final runs

    # Load results
    res = spotpy.analyser.load_csv_results(f'hbv_cal_{catch}')
    best = res[np.argmax(res['like1'])]
    best_params = {name[3:]: best[name] for name in best.dtype.names if name.startswith("par")}

    # Run model with best params
    Q_sim_cal = run_hbv_rsm(df_cal["precipitation(mm/d)"].values,
                            df_cal["temperature_mean(degC)"].values,
                            df_cal["pet_sim(mm/d)"].values,
                            best_params)

    Q_sim_val = run_hbv_rsm(df_val["precipitation(mm/d)"].values,
                            df_val["temperature_mean(degC)"].values,
                            df_val["pet_sim(mm/d)"].values,
                            best_params)

    # Compute NSE
    def nse(sim, obs):
        return 1 - np.sum((obs-sim)**2)/np.sum((obs-np.mean(obs))**2)

    nse_cal = nse(Q_sim_cal[warmup:], df_cal["discharge_spec(mm/d)"].values[warmup:])
    nse_val = nse(Q_sim_val[warmup:], df_val["discharge_spec(mm/d)"].values[warmup:])

    # Save results
    all_results.append({
        "catchment": catch,
        "NSE_cal": nse_cal,
        "NSE_val": nse_val,
        "best_params": best_params
    })

# -------------------------
# 4️⃣ Save All Results
# -------------------------
import json
with open("hbv_multi_catchment_results.json", "w") as f:
    json.dump(all_results, f, indent=4)

print("Calibration complete for all catchments. Results saved to 'hbv_multi_catchment_results.json'")